In [ ]:
import pandas as pd
import sys
from pathlib import PurePath

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

sns.set_palette('icefire')

In [ ]:
# Import all the stats files in and get a clean description of basin-wide snow depth stats as well as diffs

In [ ]:
csv_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/spatial/mean_plots'

In [ ]:
csv_list = h.fn_list(csv_dir, '*.csv')
df_list = [pd.read_csv(csv) for csv in csv_list]
# Basin and WY
# basin_wylist = [f'{PurePath(csv).stem.split("_")[0]}_wy{(PurePath(csv).stem.split("_wy")[1])[:4]}' for csv in csv_list]
basin_wylist = [f'{PurePath(csv).stem.split("_")[0]}_{(PurePath(csv).stem.split("_")[3])}' for csv in csv_list]

# Do some processing
for i, df in enumerate(df_list):
    # Drop all columns that do not contain "mean_depth_m" in it
    df = df.filter(like='mean_depth_m')
    # Compute the depth differences with the ASO column
    ref_col = df.filter(like='ASO')
    # Subtract ref_col values from all other columns
    for col in df.columns:
        if 'ASO' not in col:
            # Do an element-wise subtraction and store as a new column
            df[f'{col}_diff'] = list(df[col] - ref_col['ASO_mean_depth_m'])
    df_list[i] = df
df


In [ ]:
1+1

In [ ]:
# Append the basin and wy to each column of the corresponding dataframe in the list
for i, df in enumerate(df_list):
    df.columns = [f'{basin_wylist[i]}_{col}' for col in df.columns]

In [ ]:
big_df = pd.concat(df_list, axis=1, keys=basin_wylist)
big_df

In [ ]:
# Generate difference dataframe by removing all columns that do not contain "_diff"
diff_df = big_df.filter(like='_diff')

# Filter out Baseline columns
diff_df = diff_df.filter(regex='^(?!.*Baseline).*')
diff_df


In [ ]:
# Compute the top 10 entries with the largest positive basin-wide mean deviations
diff_df.mean().nlargest(10)

In [ ]:
# Compute the top 10 entries with the smallest negative mean deviations
diff_df.mean().nsmallest(10)

In [ ]:
# Compute the 10 entries with the largest absolute mean deviations
diff_df.mean().abs().nlargest(10)

In [ ]:
# Compute the 10 entries with the mean deviations closest to zero
diff_df.mean().abs().nsmallest(10)

In [ ]:
fig, ax = plt.subplots()
diff_df.mean().plot(ax=ax,
                    marker='x',
                    linewidth=0,
                    )
# Rotate the xticks
plt.xticks([]);



In [ ]:
# Get the colormap
cmap = sns.color_palette('icefire')
cmap

In [ ]:
# Plot only the columns that contain mean_depth_m
for i, (basinwy, df) in enumerate(zip(basin_wylist, df_list)):
    mean_depth_cols = [col for col in df.columns if 'mean_depth_m_diff' in col]
    fig, ax = plt.subplots(1, figsize=(6, 3))
    # Modify the cmap based on the number of columns
    if len(mean_depth_cols) < 4:
        # colors = [cmap[-1], cmap[0], cmap[1]]
        colors = [cmap[-1], cmap[0], cmap[3]]
    else:
        # colors = [cmap[-1], cmap[0], cmap[3], cmap[1]]
        colors = [cmap[-1], cmap[0], cmap[1], cmap[3]]

    df.plot(ax=ax,
        y=mean_depth_cols,
        color=colors,
        alpha=0.5,
        title=f'{basinwy} Mean Snow Depth Diff',
        ylabel='Mean Snow Depth diff (m)',
    )

    df.plot(ax=ax,
        y=mean_depth_cols,
        kind='bar',
        color=colors,
        # marker='x',
        # linewidth=0,
        title=f'{basinwy} Mean Snow Depth Diff',
        ylabel='Mean Snow Depth diff (m)',
    )

    # Plot the zero line
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.grid('lightgrey', linestyle='--', linewidth=0.5)
    ax.set_ylim(-1.3, 1.3)
    # Set the xtick labels as deciles from 0 to 100
    ax.set_xticks(range(df.shape[0]))
    ax.set_xticklabels([f'{f}-{f+9}%' for f in np.arange(0, 100, 10)])
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


In [ ]:
# Plot only the columns that contain mean_depth_m and drop baseline
fig, axa = plt.subplots(6, 2, figsize=(6*2, 3*6), sharex=True, sharey=True)
for i, (basinwy, df) in enumerate(zip(basin_wylist, df_list)):
    mean_depth_cols = [col for col in df.columns if 'mean_depth_m_diff' in col and 'Baseline' not in col]
    # fig, ax = plt.subplots(1, figsize=(6, 3))
    ax = axa.flatten()[i]
    # Modify the cmap based on the number of columns
    if len(mean_depth_cols) < 3:
        colors = [cmap[0], cmap[3]]
    else:
        colors = [cmap[0], cmap[1], cmap[3]]

    df.plot(ax=ax,
        y=mean_depth_cols,
        color=colors,
        # alpha=0.5,
        title=f'{basinwy} Mean Snow Depth Diff',
        ylabel='Mean Snow Depth diff (m)',
    )

    df.plot(ax=ax,
        y=mean_depth_cols,
        kind='bar',
        color=colors,
        alpha=0.85,
        # marker='x',
        # linewidth=0,
        title=f'{basinwy} Mean Snow Depth Diff',
        ylabel='Mean Snow Depth diff (m)',
    )

    # Plot the zero line
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.grid('lightgrey', linestyle='--', linewidth=0.5)
    ax.set_ylim(-1.3, 1.3)
    # Set the xtick labels as deciles from 0 to 100
    ax.set_xticks(range(df.shape[0]))
    ax.set_xticklabels([f'{f}-{f+9}%' for f in np.arange(0, 100, 10)])
    if i == 1:
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        ax.legend().remove()
